In [1]:
!pip -q install faiss-cpu sentence-transformers rank-bm25==0.2.2

In [2]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# 1. INSTALL LIBRARIES AND IMPORT MODULES

import re
import json
import pickle
import hashlib
import time

import numpy as np
import pandas as pd
import faiss
import torch

from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# Main project directory
PROJECT_DIR = Path("/content/drive/MyDrive/NLP Project")

# Existing saved files
EMBED_DIR = PROJECT_DIR / "data/embeddings/bge_small"

FAISS_DIR = (
    PROJECT_DIR / "vector_database/faiss/bge_small"
)

BM25_DIR = (
    PROJECT_DIR / "vector_database/bm25/bge_small"
)

# Directory for retrieval results
OUTPUT_DIR = PROJECT_DIR / "data/retrieval_results/hybrid"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Four chunking strategies
STRATEGIES = [
    "fixed_size",
    "sentence_based",
    "recursive",
    "section_aware"
]

# Model configuration
MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM = 384

# Retrieval configuration
BM25_TOP_K = 20
DENSE_TOP_K = 20
HYBRID_TOP_K = 20
RRF_K = 60

# Use GPU when available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Recommended query instruction for BGE
QUERY_PREFIX = (
    "Represent this sentence for searching relevant passages: "
)

# Load the embedding model once
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

assert model.get_sentence_embedding_dimension() == EMBEDDING_DIM

print("Embedding model:", MODEL_NAME)
print("Device:", DEVICE)
print("BM25 candidates:", BM25_TOP_K)
print("Dense candidates:", DENSE_TOP_K)
print("RRF constant:", RRF_K)

# Confirm saved files exist
for strategy in STRATEGIES:

    required_files = [
        EMBED_DIR / strategy / "metadata.jsonl",
        EMBED_DIR / strategy / "embeddings.npy",
        FAISS_DIR / strategy / "index.faiss",
        BM25_DIR / strategy / "bm25_index.pkl"
    ]

    for file in required_files:
        if not file.exists():
            raise FileNotFoundError(file)

print("\nAll required files are available.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_941/36663919.py:65: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert model.get_sentence_embedding_dimension() == EMBEDDING_DIM


Embedding model: BAAI/bge-small-en-v1.5
Device: cpu
BM25 candidates: 20
Dense candidates: 20
RRF constant: 60

All required files are available.


Load and validate the saved retrieval indexes

In [4]:
# 2. INDEX LOADING AND VALIDATION

# Cache loaded indexes for repeated searches
INDEX_CACHE = {}


def tokenize(text):
    """
    Same lexical tokenizer used in BM25 indexing.
    """
    return re.findall(r"[a-z0-9]+", str(text).lower())


def metadata_fingerprint(df):
    """
    Reproduce the fingerprint used during BM25 indexing.
    Detects changes to ordered passage records.
    """

    digest = hashlib.sha256()

    for vector_id, chunk_id, article_id, text in zip(
        df["vector_id"],
        df["chunk_id"],
        df["article_id"],
        df["text"]
    ):

        record = (
            f"{int(vector_id)}\t{chunk_id}\t"
            f"{article_id}\t{text}\n"
        )

        digest.update(record.encode("utf-8"))

    return digest.hexdigest()


def load_resources(strategy):
    """
    Load FAISS, BM25, and their shared metadata.

    Each resource is loaded only once per Colab session.
    """

    if strategy not in STRATEGIES:
        raise ValueError(f"Invalid strategy: {strategy}")

    if strategy in INDEX_CACHE:
        return INDEX_CACHE[strategy]

    # Load passage metadata

    metadata = pd.read_json(
        EMBED_DIR / strategy / "metadata.jsonl",
        lines=True
    )

    required = {
        "vector_id", "chunk_id", "article_id",
        "title", "text", "page_numbers"
    }

    assert required.issubset(metadata.columns)
    assert not metadata.empty
    assert metadata["chunk_id"].is_unique

    # Ensure vector IDs match metadata row positions
    assert np.array_equal(
        metadata["vector_id"].to_numpy(dtype=np.int64),
        np.arange(len(metadata))
    )

    # Load FAISS index

    faiss_index = faiss.read_index(
        str(FAISS_DIR / strategy / "index.faiss")
    )

    assert faiss_index.ntotal == len(metadata)
    assert faiss_index.d == EMBEDDING_DIM
    assert faiss_index.metric_type == faiss.METRIC_INNER_PRODUCT

    # Spot-check FAISS vectors against saved embeddings
    embeddings = np.load(
        EMBED_DIR / strategy / "embeddings.npy",
        mmap_mode="r"
    )

    assert embeddings.shape == (
        len(metadata), EMBEDDING_DIM
    )

    for i in {0, len(metadata) // 2, len(metadata) - 1}:
        assert np.allclose(
            faiss_index.reconstruct(int(i)),
            embeddings[i],
            atol=1e-6
        ), f"FAISS vector alignment failed: {strategy}"

    del embeddings

    # Load saved BM25 index

    # Only load trusted pickle files created by your project.
    with open(
        BM25_DIR / strategy / "bm25_index.pkl",
        "rb"
    ) as file:
        bm25_data = pickle.load(file)

    bm25_index = bm25_data["bm25"]

    assert isinstance(bm25_index, BM25Okapi)
    assert bm25_data["strategy"] == strategy
    assert bm25_index.corpus_size == len(metadata)

    # Verify BM25 and metadata alignment
    assert (
        bm25_data["fingerprint"]
        == metadata_fingerprint(metadata)
    ), f"BM25 metadata mismatch: {strategy}"

    resources = {
        "faiss": faiss_index,
        "bm25": bm25_index,
        "metadata": metadata
    }

    INDEX_CACHE[strategy] = resources

    return resources


# Load all four strategies
for strategy in STRATEGIES:

    resources = load_resources(strategy)

    print(
        f"{strategy}: "
        f"{resources['faiss'].ntotal} passages validated"
    )

print("\nAll retrieval resources loaded successfully.")


fixed_size: 5703 passages validated
sentence_based: 5839 passages validated
recursive: 6106 passages validated
section_aware: 7623 passages validated

All retrieval resources loaded successfully.


Implement Reciprocal Rank Fusion

In [5]:
# 3. RECIPROCAL RANK FUSION (RRF)

def reciprocal_rank_fusion(
    bm25_hits,
    dense_hits,
    rrf_k=60
):
    """
    Combine BM25 and dense retrieval rankings.

    Input:
        bm25_hits = [(vector_id, score), ...]
        dense_hits = [(vector_id, score), ...]

    Output:
        List of (vector_id, fusion_data), sorted
        by descending RRF score.
    """

    if rrf_k <= 0:
        raise ValueError("rrf_k must be positive.")

    fused = {}

    ranking_lists = {
        "bm25": bm25_hits,
        "dense": dense_hits
    }

    for source, hits in ranking_lists.items():

        for rank, (vector_id, raw_score) in enumerate(
            hits, start=1
        ):

            vector_id = int(vector_id)

            if vector_id not in fused:
                fused[vector_id] = {
                    "rrf_score": 0.0,
                    "bm25_rank": None,
                    "dense_rank": None,
                    "bm25_score": None,
                    "dense_score": None
                }

            # Add this retriever's RRF contribution
            fused[vector_id]["rrf_score"] += (
                1.0 / (rrf_k + rank)
            )

            # Record the original rank and score
            fused[vector_id][f"{source}_rank"] = rank
            fused[vector_id][f"{source}_score"] = float(
                raw_score
            )

    # Sort by fusion score; vector_id breaks ties
    ranked = sorted(
        fused.items(),
        key=lambda item: (
            -item[1]["rrf_score"],
            item[0]
        )
    )

    return ranked


# TEST RRF MATHEMATICS

sample_bm25 = [(0, 15.0), (1, 12.0)]
sample_dense = [(1, 0.91), (0, 0.88)]

test_ranking = reciprocal_rank_fusion(
    sample_bm25,
    sample_dense,
    rrf_k=60
)

expected_score = 1 / 61 + 1 / 62

assert all(
    abs(item[1]["rrf_score"] - expected_score) < 1e-10
    for item in test_ranking
)

print("RRF function test passed.")


RRF function test passed.


Implement complete hybrid retrieval

In [6]:
# 4. COMPLETE HYBRID RETRIEVAL FUNCTION

def hybrid_search(
    question,
    strategy="recursive",
    bm25_k=BM25_TOP_K,
    dense_k=DENSE_TOP_K,
    top_k=HYBRID_TOP_K,
    rrf_k=RRF_K
):
    """
    Retrieve passages using BM25 + FAISS + RRF.

    Returns:
    - Final hybrid rank and RRF score
    - Original BM25 and FAISS ranks/scores
    - Article ID, title, year, pages
    - Supporting passage text
    """

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question cannot be empty.")

    if any(
        not isinstance(k, int) or k < 1
        for k in (bm25_k, dense_k, top_k)
    ):
        raise ValueError("Retrieval sizes must be positive integers.")

    resources = load_resources(strategy)

    bm25_index = resources["bm25"]
    faiss_index = resources["faiss"]
    metadata = resources["metadata"]

    # 1. BM25 lexical retrieval

    query_tokens = tokenize(question)

    bm25_hits = []

    if query_tokens:

        bm25_scores = bm25_index.get_scores(query_tokens)

        # Keep positive, finite lexical scores
        valid = np.flatnonzero(
            np.isfinite(bm25_scores)
            & (bm25_scores > 0)
        )

        # Sort scores in descending order
        best_indices = valid[
            np.argsort(
                -bm25_scores[valid],
                kind="stable"
            )[:bm25_k]
        ]

        bm25_hits = [
            (int(i), float(bm25_scores[i]))
            for i in best_indices
        ]

    # 2. FAISS dense semantic retrieval

    # Add instruction to question, not paper passages
    query_text = QUERY_PREFIX + question.strip()

    query_vector = model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    query_vector = np.ascontiguousarray(
        query_vector,
        dtype=np.float32
    )

    k = min(dense_k, faiss_index.ntotal)

    dense_scores, dense_indices = faiss_index.search(
        query_vector,
        k
    )

    dense_hits = [
        (int(idx), float(score))
        for idx, score in zip(
            dense_indices[0],
            dense_scores[0]
        )
        if idx >= 0 and np.isfinite(score)
    ]

    # 3. Fuse the two ranking lists using RRF

    ranked = reciprocal_rank_fusion(
        bm25_hits=bm25_hits,
        dense_hits=dense_hits,
        rrf_k=rrf_k
    )

    # 4. Retrieve source metadata

    results = []

    for rank, (vector_id, fusion) in enumerate(
        ranked[:top_k],
        start=1
    ):

        chunk = metadata.iloc[vector_id]

        # Indicate which retrievers found this chunk
        found_in = []

        if fusion["bm25_rank"] is not None:
            found_in.append("BM25")

        if fusion["dense_rank"] is not None:
            found_in.append("FAISS")

        results.append({
            "rank": rank,
            "rrf_score": fusion["rrf_score"],
            "bm25_rank": fusion["bm25_rank"],
            "bm25_score": fusion["bm25_score"],
            "dense_rank": fusion["dense_rank"],
            "dense_score": fusion["dense_score"],
            "found_in": ", ".join(found_in),
            "vector_id": vector_id,
            "chunk_id": chunk["chunk_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk.get("year"),
            "page_numbers": chunk["page_numbers"],
            "section": chunk.get("section", ""),
            "text": chunk["text"]
        })

    return pd.DataFrame(results)


print("Hybrid retrieval function initialized.")


Hybrid retrieval function initialized.


Test hybrid retrieval for all four strategies

In [ ]:
# 5. EVALUATE HYBRID RETRIEVAL WITH MULTIPLE QUESTIONS

# Questions covering different fraud detection research topics
TEST_QUESTIONS = [
    {
        "id": "Q01",
        "type": "Conceptual",
        "question": "How are graph neural networks used for financial fraud detection?"
    },
    {
        "id": "Q02",
        "type": "Factual",
        "question": "Which machine learning models are used for credit card fraud detection?"
    },
    {
        "id": "Q03",
        "type": "Factual",
        "question": "Which datasets are commonly used to evaluate credit card fraud detection models?"
    },
    {
        "id": "Q04",
        "type": "Conceptual",
        "question": "How do researchers handle class imbalance in financial fraud detection datasets?"
    },
    {
        "id": "Q05",
        "type": "Factual",
        "question": "How is XGBoost applied to credit card fraud detection?"
    },
    {
        "id": "Q06",
        "type": "Conceptual",
        "question": "What are the advantages and limitations of federated learning for fraud detection?"
    },
    {
        "id": "Q07",
        "type": "Comparison",
        "question": "How do supervised and unsupervised learning methods differ in financial fraud detection?"
    },
    {
        "id": "Q08",
        "type": "Comparison",
        "question": "How do graph neural networks compare with traditional machine learning methods for fraud detection?"
    },
    {
        "id": "Q09",
        "type": "Conceptual",
        "question": "What are the challenges of real-time financial fraud detection?"
    },
    {
        "id": "Q10",
        "type": "Comparison",
        "question": "What are the main limitations of deep learning models for financial fraud detection?"
    },
    {
        "id": "Q11",
        "type": "Conceptual",
        "question": "How do explainable AI techniques such as SHAP help interpret fraud detection models?"
    },
    {
        "id": "Q12",
        "type": "Unanswerable",
        "question": "What accuracy did financial fraud detection models achieve on the Mars Banking Dataset in 2035?"
    }
]

# Number of final hybrid passages per question
TOP_K = 10

# Store all results
all_results = []
summary_rows = []

# Load indexes into cache before testing
for strategy in STRATEGIES:
    load_resources(strategy)

# RUN ALL QUESTIONS AGAINST ALL STRATEGIES

for item in TEST_QUESTIONS:

    question_id = item["id"]
    question_type = item["type"]
    question = item["question"]

    print(f"\nTesting {question_id}: {question}")

    for strategy in STRATEGIES:

        start = time.perf_counter()

        results = hybrid_search(
            question=question,
            strategy=strategy,
            bm25_k=20,
            dense_k=20,
            top_k=TOP_K,
            rrf_k=60
        )

        latency = time.perf_counter() - start

        if results.empty:
            print(f"  {strategy}: No results")
            continue

        # Preserve question information with each result
        results.insert(0, "question_id", question_id)
        results.insert(1, "question_type", question_type)
        results.insert(2, "question", question)
        results.insert(3, "strategy", strategy)

        results["latency_seconds"] = round(latency, 4)

        all_results.append(results)

        # Record retrieval statistics
        summary_rows.append({
            "Question ID": question_id,
            "Question Type": question_type,
            "Strategy": strategy,
            "Retrieved": len(results),
            "Distinct Articles": results["article_id"].nunique(),
            "Found by Both": (
                results["found_in"] == "BM25, FAISS"
            ).sum(),
            "Latency (s)": round(latency, 4)
        })

        print(
            f"  {strategy}: "
            f"{len(results)} passages, "
            f"{results['article_id'].nunique()} articles, "
            f"{latency:.4f}s"
        )

# COMBINE AND SAVE RESULTS

if not all_results:
    raise ValueError("No passages retrieved for any question.")

detailed_df = pd.concat(
    all_results,
    ignore_index=True
)

summary_df = pd.DataFrame(summary_rows)

# Save full evidence and retrieval rankings
detailed_df.to_json(
    OUTPUT_DIR / "hybrid_multiple_questions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# Save compact retrieval statistics
summary_df.to_csv(
    OUTPUT_DIR / "hybrid_multiple_questions_summary.csv",
    index=False
)

# Save the question set
with open(
    OUTPUT_DIR / "hybrid_test_questions.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(TEST_QUESTIONS, file, indent=4)

# DISPLAY FINAL RESULTS

print("\n" + "=" * 70)
print("MULTI-QUESTION HYBRID RETRIEVAL COMPLETED")
print("=" * 70)

print("Total questions:", len(TEST_QUESTIONS))
print("Strategies tested:", len(STRATEGIES))
print("Total retrieval tests:", len(summary_df))

print("\nRETRIEVAL SUMMARY:")
display(summary_df)

# Show top 3 passages for every question and strategy
top_results = detailed_df[
    detailed_df["rank"] <= 3
]

print("\nTOP 3 RESULTS PER QUESTION AND STRATEGY:")

display(
    top_results[
        [
            "question_id",
            "strategy",
            "rank",
            "rrf_score",
            "article_id",
            "title",
            "page_numbers",
            "found_in"
        ]
    ]
)

print("\nResults saved successfully.")



Testing Q01: How are graph neural networks used for financial fraud detection?
  fixed_size: 10 passages, 8 articles, 0.2821s
  sentence_based: 10 passages, 8 articles, 0.1340s
